In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score
from imblearn.under_sampling import RandomUnderSampler

In [3]:
df = pd.read_csv("/content/drive/MyDrive/Datasets/creditcard.csv")
df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [5]:
cnt = df.isnull().sum()
cnt[cnt > 0]

,0


In [7]:
df["Class"].value_counts(normalize=True)

,proportion
Class,
0,0.998273
1,0.001727


In [15]:
X = df.drop('Class', axis=1).values
y = df['Class'].values

In [ ]:
rus = RandomUnderSampler(random_state=42)
X_balanced, y_balanced = rus.fit_resample(X, y) 

In [17]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)


787 197


In [18]:
scaler  = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

In [ ]:
class perceptron:
    def __init__(self, lr, epochs, mode):
        self.lr = lr
        self.epochs = epochs
        self.mode = mode
        self.weight = None
        self.bias = None
    
    def sigmoid(self, x):
        return 1 / (1 + np.exp(-np.clip(x, -500, 500)))
    
    def fit(self, x, y):
        n, m = x.shape
        self.weight = np.zeros(m)
        self.bias = 0
        
        for epoch in range(self.epochs):
            if self.mode == "Batch":
                z = x.dot(self.weight) + self.bias
                pred = self.sigmoid(z)
                self.weight -= self.lr * x.T.dot(pred - y) / n
                self.bias -= self.lr * (pred - y).mean()
            
            else:
                for i in range(n):
                    z = x[i].dot(self.weight) + self.bias
                    pred = self.sigmoid(z)
                    self.weight -= self.lr * (pred - y[i]) * x[i]
                    self.bias -= self.lr * (pred - y[i])
    
    def predict(self, x):
        return (self.sigmoid(x.dot(self.weight) + self.bias) >= 0.5).astype(int)

In [23]:
model_batch = perceptron(lr=0.1, epochs=100, mode='batch')
model_batch.fit(X_train, y_train)

y_pred_batch = model_batch.predict(X_test)
acc_batch = accuracy_score(y_test, y_pred_batch)
prec_batch = precision_score(y_test, y_pred_batch)
rec_batch = recall_score(y_test, y_pred_batch)
f1_batch = f1_score(y_test, y_pred_batch)

print(f"Test Accuracy : {acc_batch * 100:.2f}%")
print(f"Precision : {prec_batch * 100:.2f}%")
print(f"Recall : {rec_batch * 100:.2f}%")
print(f"F1-Score : {f1_batch * 100:.2f}%")
print(f"Correct: {sum(y_pred_batch == y_test)} / {len(y_test)}")

Test Accuracy : 94.92%
Precision : 94.90%
Recall : 94.90%
F1-Score : 94.90%
Correct: 187 / 197


In [25]:
model_sgd = perceptron(lr=0.01, epochs=100, mode='stochastic')
model_sgd.fit(X_train, y_train)

y_pred_sgd = model_sgd.predict(X_test)
acc_sgd = accuracy_score(y_test, y_pred_sgd)
prec_sgd = precision_score(y_test, y_pred_sgd)
rec_sgd = recall_score(y_test, y_pred_sgd)
f1_sgd = f1_score(y_test, y_pred_sgd)

print(f"Test Accuracy : {acc_sgd  * 100:.2f}%")
print(f"Precision : {prec_sgd  * 100:.2f}%")
print(f"Recall : {rec_sgd   * 100:.2f}%")
print(f"F1-Score : {f1_sgd    * 100:.2f}%")
print(f"Correct: {sum(y_pred_sgd == y_test)} / {len(y_test)}")

Test Accuracy : 96.45%
Precision : 97.89%
Recall : 94.90%
F1-Score : 96.37%
Correct: 190 / 197


In [27]:
print("Batch GD vs Stochastic GD")

print(f"{'Metric':<20} {'Batch GD':>20} {'Stochastic GD':>20}")
print("-" * 70)
print(f"{'Accuracy':<20} {acc_batch  * 100:>19.2f}% {acc_sgd  * 100:>19.2f}%")

print(f"{'Precision':<20} {prec_batch * 100:>19.2f}% {prec_sgd * 100:>19.2f}%")
print(f"{'Recall':<20} {rec_batch  * 100:>19.2f}% {rec_sgd  * 100:>19.2f}%")
print(f"{'F1-Score':<20} {f1_batch   * 100:>19.2f}% {f1_sgd   * 100:>19.2f}%")

Batch GD vs Stochastic GD
Metric                           Batch GD        Stochastic GD
----------------------------------------------------------------------
Accuracy                           94.92%               96.45%
Precision                          94.90%               97.89%
Recall                             94.90%               94.90%
F1-Score                           94.90%               96.37%
